In [1]:
from pathlib import Path
from tqdm import tqdm

import numpy as np
import scipy

import pickle
import tifffile

import matplotlib.pyplot as plt
import seaborn as sns
import fastplotlib as fpl

from ipywidgets import VBox, HBox, Layout, Select, IntSlider, Dropdown

import pygpcca

To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Unable to find extension: VK_EXT_swapchain_colorspace


Available devices:
✅ (default) | AMD RADV POLARIS10 (ACO) | DiscreteGPU | Vulkan | Mesa 20.3.5 (ACO)
❗ | llvmpipe (LLVM 11.0.1, 256 bits) | CPU | Vulkan | Mesa 20.3.5 (LLVM 11.0.1)
✅ | NVIDIA GeForce RTX 3080 | DiscreteGPU | Vulkan | 530.30.02
❗ | Radeon RX 570 Series (POLARIS10, DRM 3.40.0, 5.10.0-21-amd64, LLVM 11.0.1) | Unknown | OpenGL | 4.6 (Core Profile) Mesa 20.3.5


In [2]:
%matplotlib inline

In [4]:
vd1 = {
    "path": Path("/home/kushal/lab/datasets/jason/M19/D05/M19D5_MC.tif"),
    "rects": [(162, 268, 178, 288), (304, 410, 118, 228), (345, 451, 297, 407)],
    "results": list()
}

vd2 = {
    "path": Path("/home/kushal/lab/datasets/jason/61/D05/61D5_MC_mmap.tif"),
    "rects": [(132, 290, 92, 264), (296, 461, 197, 414)],
    "results": list()
}

vd3 = {
    "path": Path("/home/kushal/lab/datasets/jason/885/D05/885D5_MC.tif"),
    "rects": [(155, 307, 166, 326), (231, 383, 99, 259), (126, 278, 91, 251), (47, 210, 197, 357), (179, 328, 270, 398)],
    "results": list()
}

vids = [vd1, vd2, vd3]

- view the movie subset
- percentile filtered
- line plot to click lines and show selected raw pixel and percentile filtered
- min chi line plot
- image grid of clusters, percentile filter, intslider
- linear selector on min chi to update number of clusters
- single overlaps scatter plot with random colors, use percentile filtered cluster images

In [5]:
movie_selector = Select(
    options=[v["path"].stem for v in vids],
    description="movie: "
)

CURRENT_VID = vids[movie_selector.index]

movie_raw = tifffile.imread(CURRENT_VID["path"])

iw_movie = fpl.ImageWidget(movie, cmap="gnuplot2")

CURRENT_RECTS = CURRENT_VID["rects"]
for i, rect in enumerate(CURRENT_RECTS):
    rs = iw_movie.managed_graphics[0].add_rectangle_selector(rect, edge_color="r", edge_thickness=3, metadata=i, resizable=False)
    

rect_slider = IntSlider(value=0, min=0, max=len(CURRENT_VID["rects"]))
CURRENT_RECT = CURRENT_RECTS[rect_slider.value]

c0, c1, r0, r1 = CURRENT_RECT

sub_raw = movie_raw[:, r0:r1:2, c0:c1:2].astype(np.float32).copy()
sub = np.zeros(sub_raw.shape, dtype=np.float32)
rolling_percentile_movie(sub_raw, sub)

iw_sub = fpl.ImageWidget([sub_raw, sub], cmap="viridis")

fig_perc_filt = fpl.Figure()

fig_perc_filt[0, 0].add_line(sub_raw[:, 0, 0])

perc5 = fig_perc_filt[0, 0].add_line(np.column_stack([np.arange(sub_raw.shape[0]), np.repeat(np.percentile(sub_raw[:, 0, 0], 5), sub_raw.shape[0])]), colors="r")

@iw_sub.managed_graphics[0].add_event_handler("click")
def update_line(ev):
    col, row = ev.pick_info["index"]
    fig_perc_filt[0, 0].graphics[0].data[:, 1] = sub_raw[:, row, col]
    perc5.data[:, :-1] = np.column_stack([np.arange(sub_raw.shape[0]), np.repeat(np.percentile(sub_raw[:, row, col], 5), sub_raw.shape[0])])

def get_gpcca_path(n_clusters: int = 10):
    return f"{CURRENT_VID['path'].stem}_rect-{rect_slider.value}_clusters-{n_clusters}.pickle"

GP: pygpcca.GPCCA = pickle.load(open(get_gpcca_path(), "rb"))

n_clusters_slider = IntSlider(value=10, min=10, max=25)

fig_comps = fpl.Figure(canvas="glfw")

LABELS = GP.memberships.reshape(subset.shape[1], subset.shape[2], gpcca.memberships.shape[-1])

ix = 0
for i in range(7):
    for j in range(3):
        if ix > LABELS.shape[-1] - 1:
            break
        img = LABELS.T[ix].T
        fig_comps[0, 0].add_image(img, cmap="viridis", offset=(i * img.shape[1] + 10 * i, j * img.shape[0] + 10 * j, 0))
        ix += 1

# iw_movie = fpl.ImageWidget(sub, figure_kwargs={"canvas": "glfw"}, cmap="gnuplot2")


fig_comps.show()#, iw_movie.show()

Select(description='movie: ', options=('M19D5_MC', '61D5_MC_mmap', '885D5_MC'), value='M19D5_MC')

In [ ]:
vid = vids[0]

vid["path"]